# From a LoRaWAN record to a learning problem
**Objective:** identify the observation, information, target and learning paradigm; calculate a prediction error.
**Prerequisites:** elementary Python, packets and gateways. No waveform processing.
**Route:** read the example, run each cell, answer the exercise, then inspect the worked answer.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, SVG
ROOT = Path.cwd() if (Path.cwd() / 'data/manifest.json').exists() else Path.cwd().parent
assert (ROOT / 'data/manifest.json').exists(), 'Run from the project root or notebooks directory.'
sys.path.insert(0, str(ROOT / 'scripts'))
PILOT = ROOT / 'results/pilot_v1'
MATCHED = ROOT / 'results/transition_v2'
GUIDE = ROOT / 'results/guide_v3'


## Choose your starting point
Before opening the feedback, answer four questions:

1. What is the mean of 2, 4 and 9?
2. An array has shape `(10, 3)`: how many examples and features does it contain?
3. At 20:00, can an observed SNR from the hour ending at 23:00 be an input to a forecast issued now?
4. What does evaluating only on the fitting observations leave unknown?

<details><summary>Check your reasoning and choose a route</summary>

The mean is 5; the array has ten examples and three features; the future observation is unavailable; fitting error does not assess unseen outcomes. If arithmetic or arrays are unfamiliar, read the opening of Section III and the small arrays below. If availability or evaluation is unclear, complete Sections II and V before model selection. Readers comfortable with all four can proceed to the public-data studies. This is a starting-point check, not a proficiency score.

</details>

## Same received log, different transmission histories
Two illustrative scenarios contain the same 100 deduplicated reception records and recorded metadata. Scenario A has 100 attempted uplinks; scenario B has 200, with the extra attempts leaving no record. Before running, state whether a model that sees only the shared log could identify the reception fraction.

The attempt counts below are explanatory information supplied for the example. They are not recoverable from the common log.

In [ ]:
received=100
attempts={'A':100,'B':200}
display(pd.DataFrame({'attempts':attempts,
    'received_fraction':{name:received/count for name,count in attempts.items()}}))
assert received/attempts['A']==1 and received/attempts['B']==0.5


<details><summary>Feedback: what extra observation is needed?</summary>

Both scenarios produce the same input to the learner but different reception fractions. An aligned attempt log and a definition of reception success would distinguish them. A more flexible estimator cannot distinguish identical inputs. Frame counters require session, reset and capture assumptions before they can reconstruct missing attempts.

</details>

## 1. One packet can have several receptions
The Bonn table includes reception rows, a gateway identifier and an operational packet grouping key. Inspect which quantities repeat. The loader implements the documented fixed sample; it does not fit a model.

In [ ]:
from experiments import load_pathloss
receptions, features = load_pathloss()
display(receptions[['row_id','pkt_number','gw','rssi','snr','packet_group']].head())
display(pd.Series({'receptions':len(receptions),'packet groups':receptions.packet_group.nunique()}))


## 2. Separate input, answer and audit metadata
For localization, RSSIs are inputs, position is the answer, and packet/time identifiers support partitioning. A classifier needs a defined class. A neural architecture does not determine the supervision type.

In [ ]:
X = np.array([[-80., -100.], [-84., -96.]])
Y = np.array([[0., 0.], [100., 0.]])
display(pd.DataFrame(X, columns=['gateway_1_dBm','gateway_2_dBm']))
print('Examples, features:', X.shape, 'Target dimensions:', Y.shape)
truth = np.array([110., 114.]); prediction = np.array([112., 112.])
print('MAE dB:', np.abs(truth-prediction).mean())
print('MSE dB²:', ((truth-prediction)**2).mean())


## 3. A minimal prediction contract
Write the unit, origin time, available inputs, target and intended population. For a six-hour forecast, the future signal is a label, not an input. For packet delivery, observed receptions alone do not supply all attempted transmissions.

**Exercise:** Can a network-server feature always be used by an end-device model? Can an anomaly score prove an attack?

<details><summary>Feedback after your attempt</summary>

availability depends on where and when a decision is made; feedback can have costs. An anomaly score measures disagreement with a fitted reference, not an independently confirmed incident. A new study must specify both the information path and the outcome it can observe.

</details>

**Adaptation task:** take ten rows from a new capture and label each column as input, target, identifier, clock or audit field. Explain ambiguous fields before fitting.